# Baseline: Regresión Logística + LASSO (L1, L2, L3)

**Features:** constantes vitales de triage + acuity (ESI) + n_medications (medrecon) + chief complaint (TF-IDF 50 términos)
**Targets:** L1 (ingreso hospitalario), L2 (resultado crítico), L3 (intervención crítica)
**Split:** temporal — preprocesamiento ajustado exclusivamente sobre train

Se establece el modelo de regresión logística con regularización L1 como referencia interpretable frente a los modelos más complejos del estudio. La penalización LASSO realiza selección implícita de variables, lo que facilita la interpretación de los coeficientes resultantes.

## 1. Setup y librerías

Importaciones y configuración de estilo gráfico. Las dependencias principales son `scikit-learn` (imputación, escalado, LogReg, TF-IDF) y `scipy.sparse` para la concatenación de matrices densas y sparse.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from dotenv import load_dotenv

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

## 2. Carga de datos

Se cargan las particiones de train y validación generadas por el pipeline de split temporal. Toda transformación posterior se ajusta exclusivamente sobre train.

In [2]:
load_dotenv(dotenv_path=Path("../../.env"), override=True)
if not os.getenv("MIMIC_IV_ED_PATH"):
    load_dotenv(dotenv_path=Path(".env"), override=True)

DATA         = Path(os.getenv("MIMIC_IV_ED_PATH", ""))
PROCESSED_DIR = Path("../../data/processed")

print(f"DATA         : {DATA}  |  existe: {DATA.exists()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")

DATA         : C:\Users\cuent\Documents\UAX\TFM\TFM\data  |  existe: True
PROCESSED_DIR: C:\Users\cuent\Documents\UAX\TFM\TFM\data\processed


In [3]:
print("Cargando particiones temporales...")
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_val   = pd.read_parquet(PROCESSED_DIR / "val.parquet")

print(f"Train : {df_train.shape[0]:,} pacientes, {df_train.shape[1]} columnas")
print(f"Val   : {df_val.shape[0]:,} pacientes, {df_val.shape[1]} columnas")
print(f"\nColumnas disponibles: {df_train.columns.tolist()}")

Cargando particiones temporales...
Train : 278,320 pacientes, 21 columnas
Val   : 59,640 pacientes, 21 columnas

Columnas disponibles: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint', 'L1', 'L2', 'L3']


## 3. Ingeniería de características

### 3a. medrecon → n_medications (proxy de polifarmacia)

In [ ]:
print("Cargando medrecon...")
df_medrecon = pd.read_csv(DATA / "medrecon.csv", low_memory=False)
print(f"medrecon shape: {df_medrecon.shape}")
print(df_medrecon.head(3))

# Agregación: número de medicamentos previos por estancia
n_meds = (
    df_medrecon.groupby("stay_id")
    .size()
    .rename("n_medications")
    .reset_index()
)

# Left join sobre train y val; estancias sin medrecon reciben n_medications = 0
df_train = df_train.merge(n_meds, on="stay_id", how="left")
df_val   = df_val.merge(n_meds, on="stay_id", how="left")
df_train["n_medications"] = df_train["n_medications"].fillna(0).astype(int)
df_val["n_medications"]   = df_val["n_medications"].fillna(0).astype(int)

print(f"\nn_medications — train: media={df_train['n_medications'].mean():.2f}, mediana={df_train['n_medications'].median():.0f}")
print(f"Estancias sin medicación previa (train): {(df_train['n_medications']==0).sum():,} ({(df_train['n_medications']==0).mean()*100:.1f}%)")

### 3b. chiefcomplaint → TF-IDF (ajustado solo sobre train)

Vectorización TF-IDF con vocabulario de 50 términos. El `fit` se aplica exclusivamente sobre los textos de train para evitar data leakage; `val` se transforma con el vocabulario ya ajustado.

In [ ]:
# Imputación de NaN con cadena vacía antes de vectorizar
train_text = df_train["chiefcomplaint"].fillna("").astype(str).str.lower()
val_text   = df_val["chiefcomplaint"].fillna("").astype(str).str.lower()

print(f"chiefcomplaint — % nulo train: {df_train['chiefcomplaint'].isna().mean()*100:.2f}%")
print("Motivos más frecuentes (top 10):")
print(df_train["chiefcomplaint"].value_counts().head(10))

# TF-IDF con 50 términos; fit aplicado exclusivamente sobre train
tfidf = TfidfVectorizer(max_features=50, token_pattern=r"(?u)\b\w+\b")
X_text_train = tfidf.fit_transform(train_text)   # fit + transform sobre train
X_text_val   = tfidf.transform(val_text)          # solo transform sobre val

print(f"\nVocabulario TF-IDF ({tfidf.max_features} términos):")
print(tfidf.get_feature_names_out())

### 3c. Features numéricas + matriz combinada

Imputación (mediana) y escalado (Z-score) ajustados sobre train. Se concatenan las features numéricas densas con la matriz TF-IDF sparse mediante `scipy.sparse.hstack` para obtener una única matriz de entrada al clasificador.

In [ ]:
NUMERIC_FEATURES = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity", "n_medications", "pain"]
CAT_FEATURES     = ["arrival_transport"]
TARGETS = ["L1", "L2", "L3"]

# Imputación y escalado ajustados exclusivamente sobre train
imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()

X_num_train = scaler.fit_transform(imputer.fit_transform(df_train[NUMERIC_FEATURES]))
X_num_val   = scaler.transform(imputer.transform(df_val[NUMERIC_FEATURES]))

# One-hot encoding de arrival_transport; ajustado sobre train
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat_train = ohe.fit_transform(df_train[CAT_FEATURES].fillna("desconocido"))
X_cat_val   = ohe.transform(df_val[CAT_FEATURES].fillna("desconocido"))

# Concatenación de features numéricas (densas), OHE (sparse) y TF-IDF (sparse)
X_train_full = sp.hstack([sp.csr_matrix(X_num_train), X_cat_train, X_text_train])
X_val_full   = sp.hstack([sp.csr_matrix(X_num_val), X_cat_val, X_text_val])

print(f"Matriz de features train: {X_train_full.shape}")
print(f"Matriz de features val  : {X_val_full.shape}")
print(f"  → {len(NUMERIC_FEATURES)} numéricas + {len(ohe.get_feature_names_out())} OHE (arrival_transport) + {tfidf.max_features} TF-IDF")

## 4. Entrenamiento multi-label: L1, L2, L3

Se entrena un clasificador independiente por target. Se emplea regularización L1 (LASSO) con solver `saga` y pesos de clase balanceados para compensar el desbalance en L2 y L3.

In [ ]:
TARGET_NAMES = {
    "L1": "Ingreso hospitalario",
    "L2": "Resultado crítico en urgencias",
    "L3": "Intervención crítica en urgencias",
}

results = {}
models  = {}

for target in TARGETS:
    prevalencia = df_train[target].mean() * 100
    print(f"\n{'='*60}")
    print(f"TARGET: {target} — {TARGET_NAMES[target]}")
    print(f"Prevalencia en train: {prevalencia:.2f}%")
    print(f"{'='*60}")

    y_train_t = df_train[target]
    y_val_t   = df_val[target]

    # l1_ratio=1.0 equivale a penalty='l1'; evita FutureWarning en sklearn >= 1.8
    clf = LogisticRegression(
        solver="saga",
        l1_ratio=1.0,
        class_weight="balanced",
        random_state=42,
        max_iter=2000,
    )
    clf.fit(X_train_full, y_train_t)

    y_pred = clf.predict_proba(X_val_full)[:, 1]

    auroc = roc_auc_score(y_val_t, y_pred)
    auprc = average_precision_score(y_val_t, y_pred)
    brier = brier_score_loss(y_val_t, y_pred)

    print(f"AUROC : {auroc:.4f}")
    print(f"AUPRC : {auprc:.4f}  (azar: {df_val[target].mean():.4f})")
    print(f"Brier : {brier:.4f}")

    results[target] = {"AUROC": auroc, "AUPRC": auprc, "Brier": brier, "Prev_val": df_val[target].mean()}
    models[target]  = clf

print("\nEntrenamiento completado para L1, L2 y L3.")

## 5. Tabla comparativa de resultados

El AUROC supera 0.80 en los tres targets. El AUPRC de L2 y L3 es bajo en términos absolutos pero el lift sobre el azar es sustancial (×6.5 y ×11.6 respectivamente), lo que refleja el efecto del desbalance severo de clases sobre esta métrica.

In [ ]:
df_results = pd.DataFrame(results).T
df_results.index.name = "Target"
df_results["AUPRC_lift"] = (df_results["AUPRC"] / df_results["Prev_val"]).round(2)

print("=== Regresión Logística + LASSO — Resultados en Validación ===")
print(df_results.round(4).to_string())
print("\n(AUPRC_lift = AUPRC / prevalencia — cuánto supera al azar)")

## 6. Importancia de variables por target

Los coeficientes LASSO tras la convergencia reflejan tanto la magnitud como la dirección del efecto de cada variable. Coeficientes positivos incrementan el riesgo; negativos lo reducen. La regularización L1 anula variables irrelevantes, dejando únicamente las que aportan información discriminante.

In [ ]:
# Nombres de todas las features (numéricas + OHE + TF-IDF)
feature_names = NUMERIC_FEATURES + list(ohe.get_feature_names_out()) + list(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes, TARGETS):
    coefs = models[target].coef_[0]
    df_coef = (
        pd.DataFrame({"feature": feature_names, "coef": coefs})
        .assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False)
        .head(15)
    )
    colors = ["#d62728" if c > 0 else "#1f77b4" for c in df_coef["coef"]]
    ax.barh(df_coef["feature"][::-1], df_coef["coef"][::-1], color=colors[::-1])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{target} — {TARGET_NAMES[target]}", fontsize=9)
    ax.set_xlabel("Coeficiente LASSO")

plt.suptitle("Top 15 coeficientes por target (rojo = aumenta riesgo, azul = reduce riesgo)", y=1.02)
plt.tight_layout()
plt.show()